# 02 — Làm sạch VOZ Corpus


## 1. Dependencies & Cấu hình

In [2]:
import re
import os
import pandas as pd
from pathlib import Path
from google.colab import userdata, drive
from huggingface_hub import login, whoami, HfApi

In [4]:
hf_token = userdata.get('HF_TOKEN')
login(token=hf_token, add_to_git_credential=False)
user_info = whoami()

print(f"Authenticated as: {user_info['name']}")

Authenticated as: AnoraLee


In [5]:
drive.mount('/content/drive')

PROJECT_DIR = Path("/content/drive/MyDrive/Hate_Speech_Detection")

DATA_DIR = PROJECT_DIR / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
MODELS_DIR = PROJECT_DIR / "models"
RESULTS_DIR = PROJECT_DIR / "results"

print(f"Project root: {PROJECT_DIR}")

Mounted at /content/drive
Project root: /content/drive/MyDrive/Hate_Speech_Detection


In [6]:
MIN_WORDS = 3
MAX_WORDS = 300

df = pd.read_csv(RAW_DIR / "tdtu_voz_corpus.csv")
print(f"Số dòng thô: {len(df):,}")

Số dòng thô: 500,000


## 2. Loại bỏ Rác (URL, HTML, khoảng trắng thừa)

In [7]:
URL_PATTERN = re.compile(r"https?://\S+|www\.\S+")
HTML_TAG_PATTERN = re.compile(r"<[^>]+>")
WHITESPACE_PATTERN = re.compile(r"\s+")

def remove_noise(text: str) -> str:
    text = str(text)
    text = URL_PATTERN.sub(" ", text)
    text = HTML_TAG_PATTERN.sub(" ", text)
    text = WHITESPACE_PATTERN.sub(" ", text).strip()
    return text

df["text"] = df["text"].apply(remove_noise)
df = df[df["text"].str.len() > 0].reset_index(drop=True)
print(f"Sau khi loại rác: {len(df):,}")

Sau khi loại rác: 500,000


## 3. Lọc theo Độ dài

In [8]:
word_counts = df["text"].str.split().str.len()
before = len(df)

df = df[(word_counts >= MIN_WORDS) & (word_counts <= MAX_WORDS)].reset_index(drop=True)
print(f"Lọc độ dài ({MIN_WORDS}-{MAX_WORDS} từ): {before:,} -> {len(df):,}")

Lọc độ dài (3-300 từ): 500,000 -> 485,049


## 4. Loại Trùng lặp

In [9]:
before = len(df)
df = df.drop_duplicates(subset=["text"]).reset_index(drop=True)
print(f"Loại trùng lặp: {before:,} -> {len(df):,} (loại {before - len(df):,} dòng, ~{(before - len(df)) / before:.1%})")

Loại trùng lặp: 485,049 -> 410,362 (loại 74,687 dòng, ~15.4%)


## 5. Lưu kết quả

In [10]:
df.to_csv(RAW_DIR / "voz_cleaned.csv", index=False, encoding="utf-8-sig")
print(f"Đã lưu {len(df):,} dòng vào {RAW_DIR / 'voz_cleaned.csv'}")

Đã lưu 410,362 dòng vào /content/drive/MyDrive/Hate_Speech_Detection/data/raw/voz_cleaned.csv


## 6. Load Dữ liệu Thô & Gộp Schema

In [ ]:
voz = pd.read_csv(RAW_DIR / "voz_cleaned.csv")
ytb = pd.read_csv(RAW_DIR / "tdtu_youtube.csv")

voz["source"] = "voz"
ytb["source"] = "youtube"

if "comment" in ytb.columns and "text" not in ytb.columns:
    ytb = ytb.rename(columns={"comment": "text"})

if "topic" not in ytb.columns:
    ytb["topic"] = ""
voz["topic"] = ""

pool_raw = pd.concat([voz[["text", "source", "topic"]], ytb[["text", "source", "topic"]]], ignore_index=True)
pool_raw = pool_raw.dropna(subset=["text"]).reset_index(drop=True)
print(f"Tổng số dòng thô (VOZ + YouTube): {len(pool_raw):,}")

## 7. Chuẩn hoá: Unicode NFC + Teencode

In [ ]:
!pip install -q py_vncorenlp

import os
import re
import unicodedata
import py_vncorenlp

# Tải và khởi tạo VnCoreNLP
model_dir = '/content/vncorenlp'
if not os.path.exists(model_dir):
    py_vncorenlp.download_model(save_dir=model_dir)
rdrsegmenter = py_vncorenlp.VnCoreNLP(annotators=["wseg"], save_dir=model_dir)

# Từ điển Teencode & Toxic
TOXIC_TEENCODE_MAP = {
    r"\bko\b": "không", r"\bhok\b": "không", r"\bdc\b": "được", r"\bđc\b": "được",
    r"\bj\b": "gì", r"\bbt\b": "bình thường", r"\btrc\b": "trước", r"\bnhg\b": "nhưng",
    r"\bdm\b": "địt mẹ", r"\bđm\b": "địt mẹ", r"\bdkm\b": "địt con mẹ", r"\bđkm\b": "địt con mẹ",
    r"\bvkl\b": "vãi cả lồn", r"\bvcl\b": "vãi cả lồn", r"\bvl\b": "vãi lồn",
    r"\bcc\b": "cục cứt", r"\bcđm\b": "con đĩ mẹ", r"\bml\b": "mặt lồn", r"\bcc\b": "con cặc",
    r"\bđjt\b": "địt", r"\bdjt\b": "địt", r"\bdit\b": "địt",
    r"\bloz\b": "lồn", r"\blon\b": "lồn", r"\bcac\b": "cặc", r"\bcặk\b": "cặc",
}

def normalize_unicode(text):
    return unicodedata.normalize("NFC", str(text))

def normalize_teencode(text):
    for pattern, replacement in TOXIC_TEENCODE_MAP.items():
        text = re.sub(pattern, replacement, text, flags=re.IGNORECASE)
    return text

def segment_text_vncorenlp(text):
    try:
        sentences = rdrsegmenter.word_segment(text)
        return " ".join([" ".join(sentence) for sentence in sentences])
    except:
        return text

# CẬP NHẬT TRỰC TIẾP VÀO POOL
# 1. Chạy NFC và Teencode
pool_raw["text_raw"] = pool_raw["text"].apply(normalize_unicode).apply(normalize_teencode)

# 2. Tạo bản text đã phân từ
pool_raw["text"] = pool_raw["text_raw"].apply(segment_text_vncorenlp)

# 3. Loại trùng dựa trên text_raw
pool_raw = pool_raw.drop_duplicates(subset=["text_raw"]).reset_index(drop=True)
print(f"Sau khi chuẩn hoá + loại trùng nội bộ: {len(pool_raw):,}")

## 8. Lọc thô bằng Từ khoá

In [ ]:
COARSE_OFFENSIVE_KEYWORDS = [
    "ngu", "óc chó", "súc vật", "đồ chó", "con chó", "khốn nạn",
    "vô học", "mất dạy", "đĩ", "điếm", "thằng ngu", "con ngu",
    "đm", "đéo", "địt",
]

pattern = "|".join(re.escape(kw) for kw in COARSE_OFFENSIVE_KEYWORDS)
keyword_mask = pool_raw["text"].str.lower().str.contains(pattern, regex=True, na=False)

candidate_pool = pool_raw[keyword_mask].reset_index(drop=True)
print(f"Số dòng nghi ngờ OFFENSIVE/HATE sau lọc từ khoá: {len(candidate_pool):,} "
      f"({len(candidate_pool) / len(pool_raw):.1%} tổng dữ liệu)")

## 9. Xác nhận Nhãn bằng ViHateT5

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(HSD_MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(HSD_MODEL_NAME).to(device)
model.eval()

TASK_PREFIX = "hate-speech-detection: "
LABEL_MAP = {"clean": "CLEAN", "offensive": "OFFENSIVE", "hate": "HATE"}
CHECKPOINT_PATH = PROCESSED_DIR / "_retrieval_pool_labeling_checkpoint.csv"
CHECKPOINT_EVERY = 20

if CHECKPOINT_PATH.exists():
    CHECKPOINT_PATH.unlink()
    print("Đã xoá checkpoint cũ (nhãn rác do thiếu prefix).")

def predict_labels(texts, batch_size=32, checkpoint_path=CHECKPOINT_PATH, checkpoint_every=CHECKPOINT_EVERY):
    start_idx = 0
    predictions = []

    if checkpoint_path.exists():
        saved = pd.read_csv(checkpoint_path)
        predictions = saved["label"].tolist()
        start_idx = len(predictions)
        print(f"Tìm thấy checkpoint -- đã predict {start_idx:,}/{len(texts):,} dòng, tiếp tục từ đó.")

    for batch_num, i in enumerate(range(start_idx, len(texts), batch_size)):
        batch = [TASK_PREFIX + t for t in texts[i:i + batch_size]]
        inputs = tokenizer(batch, return_tensors="pt", truncation=True, padding=True, max_length=128).to(device)
        with torch.no_grad():
            output_ids = model.generate(**inputs, max_new_tokens=8)
        decoded = tokenizer.batch_decode(output_ids, skip_special_tokens=True)
        predictions.extend([LABEL_MAP.get(d.strip().lower(), d.strip().upper()) for d in decoded])

        if (batch_num + 1) % checkpoint_every == 0 or i + batch_size >= len(texts):
            pd.DataFrame({"label": predictions}).to_csv(checkpoint_path, index=False)
            print(f"  Checkpoint: {len(predictions):,}/{len(texts):,} dòng đã predict")

    return predictions

### 9a. Xác minh định dạng output thô (bắt buộc chạy trước khi predict cả batch)

In [ ]:
sample_texts = candidate_pool["text"].head(20).tolist()
sample_prefixed = [TASK_PREFIX + t for t in sample_texts]  # thêm prefix trước khi tokenize
sample_inputs = tokenizer(sample_prefixed, return_tensors="pt", truncation=True, padding=True, max_length=128).to(device)
with torch.no_grad():
    sample_output_ids = model.generate(**sample_inputs, max_new_tokens=8)
sample_decoded = tokenizer.batch_decode(sample_output_ids, skip_special_tokens=True)

print("Output thô của model (kiểm tra xem có khớp giả định 'clean'/'offensive'/'hate' không):")
for text, decoded in zip(sample_texts, sample_decoded):
    print(f"  {decoded!r:20s} <- {text[:60]}")

### 9b. Chạy predict trên toàn bộ candidate_pool

In [ ]:
candidate_pool["label"] = predict_labels(candidate_pool["text"].tolist())
print(candidate_pool["label"].value_counts())

if CHECKPOINT_PATH.exists():
    CHECKPOINT_PATH.unlink()
    print("Đã xoá checkpoint tạm (predict hoàn tất).")

In [ ]:
pool_labeled = candidate_pool[candidate_pool["label"].isin(["OFFENSIVE", "HATE"])].reset_index(drop=True)
print(f"Giữ lại sau khi loại dự đoán CLEAN: {len(pool_labeled):,}")
print(pool_labeled["label"].value_counts())

## 10. Phân từ (để nhất quán với train)

In [ ]:
def segment_text(text):
    return word_tokenize(text, format="text")

pool_labeled["text_raw"] = pool_labeled["text"]
pool_labeled["text"] = pool_labeled["text_raw"].apply(segment_text)
print("Đã phân từ. Ví dụ:")
print(pool_labeled[["text_raw", "text"]].iloc[0].to_dict())

## 11. Soát mẫu Thủ công (bắt buộc)

In [ ]:
sample_for_review = pool_labeled.sample(min(20, len(pool_labeled)), random_state=42)[["text_raw", "label", "source"]]
for _, row in sample_for_review.iterrows():
    print(f"[{row['label']}] ({row['source']}) {row['text_raw']}")

## 12. Lưu kết quả

In [ ]:
pool_labeled[["text", "text_raw", "label", "source", "topic"]].to_csv(
    PROCESSED_DIR / "retrieval_pool_labeled.csv", index=False, encoding="utf-8-sig"
)
print(f"Đã lưu {len(pool_labeled):,} dòng vào {PROCESSED_DIR / 'retrieval_pool_labeled.csv'}")